# Valuación Fundamental de Aluar Aluminio Argentino S.A.I.C. (ALUA.BA)

**Notebook Maestro Consolidado Autocontenido — Módulos M1 a M13**

*Cátedra de Economía y Técnica Bursátil · FCE UNCuyo*\
*Analista: Federico Agustín Chillón*

---

Este notebook ejecuta de manera **100% autónoma e inline** la totalidad del modelo cuantitativo de valuación sin requerir módulos externos. Todos los datos, funciones y cálculos están integrados en las celdas a continuación.

## Celda 0: Importaciones Estándar de Python

In [ ]:
# Celda 0: Importaciones Estándar de Python Científico
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, Image

# Configuración gráfica global (Goldman Sachs / UNCuyo Style)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#CCCCCC'
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['grid.color'] = '#EAEAEA'
plt.rcParams['grid.linestyle'] = '--'

# Directorio de datos y resultados
# Directorio portable: la carpeta donde vive este notebook (03_Modelo_y_Codigo/),
# que ya contiene static_inputs.json, resultados_original.json y muestra_montecarlo.npy.
# Robusto a que el notebook se lance desde su propia carpeta o desde la raiz del repo.
WORK_DIR = os.getcwd()
if not os.path.exists(os.path.join(WORK_DIR, "static_inputs.json")):
    _candidato = os.path.join(WORK_DIR, "03_Modelo_y_Codigo")
    if os.path.exists(os.path.join(_candidato, "static_inputs.json")):
        WORK_DIR = _candidato
print("Directorio de trabajo:", WORK_DIR)


## Módulo M0: Datos Auditados (Base de Datos Histórica FY2020–FY2025)

In [ ]:
# M0 -- Base de Datos Auditada FY2020-FY2025 y Parámetros del Modelo
# -*- coding: utf-8 -*-
"""
datos_auditados.py
==================
Estados Financieros Consolidados de ALUAR ALUMINIO ARGENTINO S.A.I.C.
FY2020 a FY2025 (ejercicios cerrados al 30 de junio), auditados por
Price Waterhouse & Co. S.R.L.

CRITERIO DE MONEDA (NIC 29)
---------------------------
Cada ejercicio se toma de la columna del ejercicio CORRIENTE de SU PROPIO
informe anual, es decir, expresado en moneda de cierre de ESE ejercicio.
NO se usan las columnas comparativas, porque bajo NIC 29 el comparativo se
reexpresa a la moneda de cierre del ejercicio siguiente y por lo tanto no es
homogeneo con el tipo de cambio de su propio cierre.

  Verificacion del criterio: el informe FY2022 muestra un total de activo al
  30.06.2021 de $188.861.630.085, mientras que el informe FY2021 muestra al
  30.06.2021 un total de $115.171.762.867. El cociente (1,6398) es el
  coeficiente de reexpresion por inflacion del ejercicio, no una variacion
  real de la magnitud.

CORRECCION RESPECTO DEL MODELO ANTERIOR
---------------------------------------
canonical_financials.json (modelo previo) tomo FY2020 de la columna
COMPARATIVA del informe FY2021, ya reexpresada a moneda de junio de 2021,
y la dividio por el CCL de junio de 2020 (77). Eso sobrestima todas las
magnitudes de FY2020 en dolares por el factor de reexpresion 1,5020.

  Ventas FY2020 segun columna comparativa FY2021 : $95.240.890.476  -> USD 1.236,9 MM
  Ventas FY2020 segun informe propio FY2020      : $63.409.348.048  -> USD   823,5 MM

Este archivo usa la cifra del informe propio FY2020. Todos los demas
ejercicios ya estaban tomados de su propio informe y se verificaron uno por uno.

FUENTES
-------
  FY2020: Aluar Jun_2020..pdf, pp. 47/48/51 (texto nativo)
  FY2021: Aluar Jun_2021 Consolidado..pdf, pp. 4/5/8 (texto nativo)
  FY2022: Aluar Jun_2022 Consolidado..pdf, pp. 3/5/8 (texto nativo)
  FY2023: Aluar Jun_2023 Consolidado..pdf, pp. 4/5/8 (texto nativo)
  FY2024: Aluar Jun_2024 Consolidado..pdf, pp. 4/5/8 (OCR Tesseract 400 dpi)
  FY2025: Aluar Jun_2025 Consolidado..pdf, pp. 3/4/7 (OCR Tesseract 400 dpi)

Toda cifra OCR se valido exigiendo que los subtotales sumen exactamente
(activo corriente + no corriente = total del activo; pasivo + patrimonio =
total del activo; EBIT + resultado financiero + asociadas = resultado antes
de impuestos; resultado antes de impuestos - impuesto = resultado del
ejercicio; FCO + FCI + FCF = variacion neta del efectivo).

TIPO DE CAMBIO
--------------
ccl_cierre = Contado con Liquidacion implicito de junio de cada ejercicio.
Se mantienen los valores del modelo previo (canonical_financials.json) para
no introducir diferencias ajenas al cambio metodologico.
"""

# Contado con Liquidacion de cierre de cada ejercicio (30 de junio)
CCL_CIERRE = {2020: 77.0, 2021: 165.0, 2022: 263.0, 2023: 503.0, 2024: 1350.0, 2025: 1430.0}

ACCIONES_EN_CIRCULACION = 2_800_000_000  # EEFF, Resultado por accion

# ---------------------------------------------------------------------------
# ESTADO DE RESULTADOS INTEGRALES CONSOLIDADO (en pesos, moneda de cierre)
# ---------------------------------------------------------------------------
ESTADO_RESULTADOS = {
    2020: dict(
        ventas_netas             =  63_409_348_048,
        costo_ventas             = -54_760_342_500,
        resultado_bruto          =   8_649_005_548,
        otros_resultados_op      =               0,
        costos_distribucion      =  -3_208_310_641,
        gastos_administracion    =  -1_706_887_987,
        otras_ganancias_perdidas =     -30_171_482,
        resultado_operativo      =   3_703_635_438,
        resultado_financiero     =  -6_566_063_005,
        resultado_asociadas      =      55_902_016,
        resultado_antes_imp      =  -2_806_525_551,
        impuesto_ganancias       =    -576_987_122,
        resultado_ejercicio      =  -3_383_512_673,
        resultado_accionistas    =  -3_657_833_689,
    ),
    2021: dict(
        ventas_netas             =  84_727_990_356,
        costo_ventas             = -68_373_389_110,
        resultado_bruto          =  16_354_601_246,
        otros_resultados_op      =               0,
        costos_distribucion      =  -3_652_477_359,
        gastos_administracion    =  -2_587_422_482,
        otras_ganancias_perdidas =       4_335_635,
        resultado_operativo      =  10_119_037_040,
        resultado_financiero     =   2_419_331_881,
        resultado_asociadas      =       8_263_359,
        resultado_antes_imp      =  12_546_632_280,
        impuesto_ganancias       =  -7_905_027_168,
        resultado_ejercicio      =   4_641_605_112,
        resultado_accionistas    =   4_872_789_993,
    ),
    2022: dict(
        ventas_netas             = 172_191_945_319,
        costo_ventas             =-117_898_237_133,
        resultado_bruto          =  54_293_708_186,
        otros_resultados_op      =               0,
        costos_distribucion      =  -7_072_608_359,
        gastos_administracion    =  -4_514_399_010,
        otras_ganancias_perdidas =     -63_699_146,
        resultado_operativo      =  42_643_001_671,
        resultado_financiero     =   9_632_037_259,
        resultado_asociadas      =    -110_423_361,
        resultado_antes_imp      =  52_164_615_569,
        impuesto_ganancias       = -21_779_035_241,
        resultado_ejercicio      =  30_385_580_328,
        resultado_accionistas    =  29_986_923_221,
    ),
    2023: dict(
        ventas_netas             = 325_816_454_929,
        costo_ventas             =-269_627_442_436,
        resultado_bruto          =  56_189_012_493,
        otros_resultados_op      =               0,
        costos_distribucion      = -14_244_626_233,
        gastos_administracion    = -10_963_048_835,
        otras_ganancias_perdidas =     187_212_424,
        resultado_operativo      =  31_168_549_849,
        resultado_financiero     =  44_564_001_050,
        resultado_asociadas      =    -109_126_134,
        resultado_antes_imp      =  75_623_424_765,
        impuesto_ganancias       =  -5_700_597_582,
        resultado_ejercicio      =  69_922_827_183,
        resultado_accionistas    =  66_183_247_107,
    ),
    2024: dict(
        ventas_netas             =1_236_401_592_941,
        costo_ventas             =-1_004_361_399_758,
        resultado_bruto          =  232_040_193_183,
        otros_resultados_op      =   69_358_315_372,
        costos_distribucion      =  -55_831_798_811,
        gastos_administracion    =  -38_755_109_755,
        otras_ganancias_perdidas =     -845_996_116,
        resultado_operativo      =  205_965_603_873,
        resultado_financiero     =    4_962_718_527,
        resultado_asociadas      =      300_555_040,
        resultado_antes_imp      =  211_228_877_440,
        impuesto_ganancias       =  -90_078_786_044,
        resultado_ejercicio      =  121_150_091_396,
        resultado_accionistas    =  122_054_902_973,
    ),
    2025: dict(
        ventas_netas             =1_562_412_110_649,
        costo_ventas             =-1_330_677_784_788,
        resultado_bruto          =  231_734_325_861,
        otros_resultados_op      =   43_157_521_696,
        costos_distribucion      =  -83_210_844_083,
        gastos_administracion    =  -60_925_022_982,
        otras_ganancias_perdidas =    1_186_589_783,
        resultado_operativo      =  131_942_570_275,
        resultado_financiero     =  -48_050_930_393,
        resultado_asociadas      =     -256_815_924,
        resultado_antes_imp      =   83_634_823_958,
        impuesto_ganancias       =  -70_518_972_115,
        resultado_ejercicio      =   13_115_851_843,
        resultado_accionistas    =    7_513_255_855,
    ),
}

# ---------------------------------------------------------------------------
# ESTADO DE SITUACION FINANCIERA CONSOLIDADO (en pesos, moneda de cierre)
# ---------------------------------------------------------------------------
BALANCE = {
    2020: dict(
        efectivo                  =  1_839_369_042,
        cuentas_por_cobrar        =  2_963_245_410,
        inventarios               = 25_148_031_793,
        otros_creditos_c          =  2_825_305_007,
        creditos_impositivos_c    =  2_191_062_408,
        otros_activos_fin_c       =     36_642_836,
        otras_inversiones_c       =              0,
        activo_corriente          = 35_003_656_496,
        ppe                       = 44_175_210_057,
        intangibles               =  1_383_549_875,
        inversiones_asociadas     =    179_154_769,
        otros_creditos_nc         =    222_412_710,
        creditos_impositivos_nc   =     19_116_441,
        activo_imp_diferido       =     22_050_318,
        activo_no_corriente       = 46_001_494_170,
        total_activo              = 81_005_150_666,
        cuentas_por_pagar_c       =  4_615_356_124,
        deuda_financiera_c        = 11_531_280_521,
        otros_pasivos_c           =  2_337_811_076,
        pasivo_corriente          = 18_484_447_721,
        cuentas_por_pagar_nc      =    623_801_384,
        deuda_financiera_nc       = 17_358_044_398,
        otros_pasivos_nc          =  8_868_653_577,
        pasivo_no_corriente       = 26_850_499_359,
        total_pasivo              = 45_334_947_080,
        total_patrimonio          = 35_670_203_586,
    ),
    2021: dict(
        efectivo                  =  7_252_917_427,
        cuentas_por_cobrar        =  3_689_394_219,
        inventarios               = 36_480_731_995,
        otros_creditos_c          =  3_668_735_416,
        creditos_impositivos_c    =  1_399_009_335,
        otros_activos_fin_c       =        910_837,
        otras_inversiones_c       =              0,
        activo_corriente          = 52_491_699_229,
        ppe                       = 60_199_672_377,
        intangibles               =  1_690_086_480,
        inversiones_asociadas     =    277_353_962,
        otros_creditos_nc         =    314_326_591,
        creditos_impositivos_nc   =    198_624_228,
        activo_imp_diferido       =              0,
        activo_no_corriente       = 62_680_063_638,
        total_activo              =115_171_762_867,
        cuentas_por_pagar_c       =  4_789_321_872,
        deuda_financiera_c        =  6_119_068_136,
        otros_pasivos_c           =  2_990_271_080,
        pasivo_corriente          = 13_898_661_088,
        cuentas_por_pagar_nc      =              0,
        deuda_financiera_nc       = 23_811_203_685,
        otros_pasivos_nc          = 19_732_650_053,
        pasivo_no_corriente       = 43_543_853_738,
        total_pasivo              = 57_442_514_826,
        total_patrimonio          = 57_729_248_041,
    ),
    2022: dict(
        efectivo                  = 20_161_189_895,
        cuentas_por_cobrar        = 18_917_452_767,
        inventarios               = 77_031_996_402,
        otros_creditos_c          =  8_736_893_481,
        creditos_impositivos_c    =  1_104_585_898,
        otros_activos_fin_c       =      1_598_589,
        otras_inversiones_c       =              0,
        activo_corriente          =125_953_717_032,
        ppe                       = 89_916_231_377,
        intangibles               =  2_135_899_711,
        inversiones_asociadas     =    344_388_827,
        otros_creditos_nc         =    304_375_359,
        creditos_impositivos_nc   =    307_457_623,
        activo_imp_diferido       =              0,
        activo_no_corriente       = 93_008_352_897,
        total_activo              =218_962_069_929,
        cuentas_por_pagar_c       = 11_888_382_325,
        deuda_financiera_c        = 11_438_716_791,
        otros_pasivos_c           = 19_755_544_912,
        pasivo_corriente          = 43_082_644_028,
        cuentas_por_pagar_nc      =              0,
        deuda_financiera_nc       = 23_801_570_192,
        otros_pasivos_nc          = 32_016_679_052,
        pasivo_no_corriente       = 55_818_249_244,
        total_pasivo              = 98_900_893_272,
        total_patrimonio          =120_061_176_657,
    ),
    2023: dict(
        efectivo                  = 29_693_029_170,
        cuentas_por_cobrar        = 15_464_292_214,
        inventarios               =181_793_434_471,
        otros_creditos_c          =  6_533_666_474,
        creditos_impositivos_c    = 13_030_551_476,
        otros_activos_fin_c       =      4_331_815,
        otras_inversiones_c       =              0,
        activo_corriente          =246_519_305_620,
        ppe                       =207_445_737_487,
        intangibles               =  3_242_978_064,
        inversiones_asociadas     =    633_308_241,
        otros_creditos_nc         =    889_239_241,
        creditos_impositivos_nc   =    162_765_479,
        activo_imp_diferido       =     51_563_967,
        activo_no_corriente       =212_425_592_479,
        total_activo              =458_944_898_099,
        cuentas_por_pagar_c       = 29_364_578_190,
        deuda_financiera_c        = 45_532_934_684,
        otros_pasivos_c           =  9_491_032_148,
        pasivo_corriente          = 84_388_545_022,
        cuentas_por_pagar_nc      =              0,
        deuda_financiera_nc       = 69_611_275_176,
        otros_pasivos_nc          = 31_991_505_856,
        pasivo_no_corriente       =101_602_781_032,
        total_pasivo              =185_991_326_054,
        total_patrimonio          =272_953_572_045,
    ),
    2024: dict(
        efectivo                  =  266_322_192_658,
        cuentas_por_cobrar        =   88_423_622_080,
        inventarios               =  756_454_320_091,
        otros_creditos_c          =   77_810_698_453,
        creditos_impositivos_c    =   18_080_070_469,
        otros_activos_fin_c       =   11_773_861_948,
        otras_inversiones_c       =                0,
        activo_corriente          =1_218_864_765_699,
        ppe                       =  737_899_921_734,
        intangibles               =    6_926_605_491,
        inversiones_asociadas     =    2_653_483_256,
        otros_creditos_nc         =    2_808_147_415,
        creditos_impositivos_nc   =       94_863_121,
        activo_imp_diferido       =    1_079_727_342,
        activo_no_corriente       =  751_462_748_359,
        total_activo              =1_970_327_514_058,
        cuentas_por_pagar_c       =  107_703_695_971,
        deuda_financiera_c        =   83_358_417_754,
        otros_pasivos_c           =   56_316_326_894,
        pasivo_corriente          =  247_378_440_619,
        cuentas_por_pagar_nc      =                0,
        deuda_financiera_nc       =  468_120_057_201,
        otros_pasivos_nc          =  120_279_464_099,
        pasivo_no_corriente       =  588_399_521_300,
        total_pasivo              =  835_777_961_919,
        total_patrimonio          =1_134_549_552_139,
    ),
    2025: dict(
        efectivo                  =   98_947_674_124,
        cuentas_por_cobrar        =   80_168_140_299,
        inventarios               =1_142_056_656_056,
        otros_creditos_c          =   54_742_518_963,
        creditos_impositivos_c    =   42_823_904_159,
        otros_activos_fin_c       =   19_404_759_127,
        otras_inversiones_c       =   32_020_256_411,
        activo_corriente          =1_470_163_909_139,
        ppe                       =1_259_939_797_125,
        intangibles               =    2_677_009_787,
        inversiones_asociadas     =    3_442_673_667,
        otros_creditos_nc         =   21_780_377_930,
        creditos_impositivos_nc   =   53_887_418_251,
        activo_imp_diferido       =    3_831_471_042,
        activo_no_corriente       =1_345_558_747_802,
        total_activo              =2_815_722_656_941,
        cuentas_por_pagar_c       =  111_743_862_956,
        deuda_financiera_c        =  433_391_263_269,
        otros_pasivos_c           =   56_229_216_361,
        pasivo_corriente          =  601_364_342_586,
        cuentas_por_pagar_nc      =                0,
        deuda_financiera_nc       =  417_391_464_490,
        otros_pasivos_nc          =  210_646_455_799,
        pasivo_no_corriente       =  628_037_920_289,
        total_pasivo              =1_229_402_262_875,
        total_patrimonio          =1_586_320_394_066,
    ),
}

# ---------------------------------------------------------------------------
# ESTADO DE FLUJO DE EFECTIVO CONSOLIDADO (en pesos, moneda de cierre)
# ---------------------------------------------------------------------------
FLUJO_EFECTIVO = {
    2020: dict(
        depreciacion       =  5_008_516_569,
        amortizacion       =    258_001_370,
        fco                = 18_920_408_390,
        capex              = -7_931_457_989,
        altas_intangibles  =              0,
        otros_inversion    =     65_838_537,
        fci                = -7_865_619_452,
        altas_deuda        = 18_779_900_162,
        cancelacion_deuda  =-26_448_369_650,
        intereses_pagados  =  2_044_293_453,
        dividendos_pagados = -5_519_136_360,
        fcf                =-15_231_899_301,
        variacion_efectivo = -4_177_110_363,
    ),
    2021: dict(
        depreciacion       =  7_680_589_089,
        amortizacion       =    388_006_517,
        fco                = 16_892_256_112,
        capex              = -1_555_450_901,
        altas_intangibles  =              0,
        otros_inversion    =              0,
        fci                = -1_555_450_901,
        altas_deuda        =  7_469_764_735,
        cancelacion_deuda  =-12_192_743_745,
        intereses_pagados  =  2_456_308_232,
        dividendos_pagados =   -265_084_677,
        fcf                = -7_444_371_919,
        variacion_efectivo =  7_892_433_292,
    ),
    2022: dict(
        depreciacion       = 11_638_889_080,
        amortizacion       =    635_547_731,
        fco                = 23_470_364_498,
        capex              = -2_921_329_114,
        altas_intangibles  =              0,
        otros_inversion    =              0,
        fci                = -2_921_329_114,
        altas_deuda        =  5_690_458_302,
        cancelacion_deuda  = -8_342_977_824,
        intereses_pagados  =  1_504_095_316,
        dividendos_pagados = -4_882_984_193,
        fcf                = -9_039_599_031,
        variacion_efectivo = 11_509_436_353,
    ),
    2023: dict(
        depreciacion       = 19_503_234_438,
        amortizacion       =  1_371_786_355,
        fco                = 57_135_968_178,
        capex              =-33_141_554_950,
        altas_intangibles  =    -10_185_981,
        otros_inversion    =              0,
        fci                =-33_151_740_931,
        altas_deuda        = 89_023_959_113,
        cancelacion_deuda  =-60_949_912_278,
        intereses_pagados  =  8_060_216_409,
        dividendos_pagados =-55_603_571_486,
        fcf                =-35_589_741_060,
        variacion_efectivo =-11_605_513_813,
    ),
    2024: dict(
        depreciacion       =  65_258_463_496,
        amortizacion       =   5_123_435_428,
        fco                = 106_907_787_111,
        capex              = -34_543_200_226,
        altas_intangibles  =      -1_414_203,
        otros_inversion    =               0,
        fci                = -34_544_614_429,
        altas_deuda        = 402_325_708_771,
        cancelacion_deuda  =-258_529_327_981,
        intereses_pagados  =  17_707_792_075,
        dividendos_pagados =  -3_133_677_232,
        fcf                = 122_954_911_483,
        variacion_efectivo = 195_318_084_165,
    ),
    2025: dict(
        depreciacion       =  94_401_647_331,
        amortizacion       =   7_109_275_850,
        fco                =  43_951_466_473,
        capex              =-348_821_496_391,
        altas_intangibles  =    -129_203_516,
        otros_inversion    =               0,
        fci                =-348_950_700_207,
        altas_deuda        = 533_524_217_717,
        cancelacion_deuda  =-458_075_268_932,
        intereses_pagados  =  37_426_104_537,
        dividendos_pagados =  -9_059_413_297,
        fcf                =  28_963_430_951,
        variacion_efectivo =-276_035_802_783,
    ),
}

# ---------------------------------------------------------------------------
# VOLUMEN FISICO — Memoria Anual, Planta de Puerto Madryn (Division Primario)
# "Total solidificado mas despacho de aluminio liquido", en toneladas.
# Es la magnitud que alimenta el modelo de Ingresos = Precio x Cantidad.
# ---------------------------------------------------------------------------
VOLUMEN_TN = {
    2020: 389_696,   # 376.085 solidificado + 13.611 liquido — Memoria 2020, p.2
    2021: 302_735,   # 287.051 solidificado + 15.684 liquido — Memoria 2021, p.2
    2022: 355_817,   # 338.929 solidificado + 16.888 liquido — Memoria 2022, p.2
    2023: 423_709,   # total informado                        — Memoria 2023, p.2
    2024: 443_425,   # 424.537 solidificado + 18.888 liquido — Memoria 2024, p.2
    2025: 442_437,   # 425.409 solidificado + 17.028 liquido — Memoria 2025, p.4
}

# Utilizacion media de la capacidad instalada informada en cada Memoria
UTILIZACION_INFORMADA = {2020: 0.8441, 2021: 0.6832, 2022: 0.7940,
                         2023: 0.9450, 2024: 0.9640, 2025: 0.9680}

CAPACIDAD_INSTALADA_TN = 460_000   # Memoria Anual — capacidad nominal de la planta

ANIOS = [2020, 2021, 2022, 2023, 2024, 2025]


def verificar():
    """Chequeos de integridad contable. Falla ruidosamente si algo no cierra."""
    errores = []
    for y in ANIOS:
        b, r, f = BALANCE[y], ESTADO_RESULTADOS[y], FLUJO_EFECTIVO[y]

        def chk(nombre, a, b_, tol=1):
            if abs(a - b_) > tol:
                errores.append(f"FY{y} {nombre}: {a:,} != {b_:,} (dif {a - b_:,})")

        chk("activo = corriente + no corriente",
            b["total_activo"], b["activo_corriente"] + b["activo_no_corriente"])
        chk("pasivo = corriente + no corriente",
            b["total_pasivo"], b["pasivo_corriente"] + b["pasivo_no_corriente"])
        chk("activo = pasivo + patrimonio",
            b["total_activo"], b["total_pasivo"] + b["total_patrimonio"])
        chk("resultado bruto", r["resultado_bruto"], r["ventas_netas"] + r["costo_ventas"])
        chk("EBIT", r["resultado_operativo"],
            r["resultado_bruto"] + r["otros_resultados_op"] + r["costos_distribucion"]
            + r["gastos_administracion"] + r["otras_ganancias_perdidas"])
        chk("resultado antes de impuestos", r["resultado_antes_imp"],
            r["resultado_operativo"] + r["resultado_financiero"] + r["resultado_asociadas"])
        chk("resultado del ejercicio", r["resultado_ejercicio"],
            r["resultado_antes_imp"] + r["impuesto_ganancias"])
        chk("variacion del efectivo", f["variacion_efectivo"], f["fco"] + f["fci"] + f["fcf"])
    return errores


if __name__ == "__main__":
    errs = verificar()
    if errs:
        print("FALLAS DE INTEGRIDAD:")
        for e in errs:
            print("  -", e)
        raise SystemExit(1)
    print("[OK] Los 6 ejercicios cierran contablemente (balance, resultados y flujo de efectivo).")
    print()
    print(f"{'Ejercicio':>10} {'CCL':>8} {'Ventas ARS':>22} {'Ventas USD MM':>15} {'EBIT USD MM':>13}")
    for y in ANIOS:
        v = ESTADO_RESULTADOS[y]["ventas_netas"]
        e = ESTADO_RESULTADOS[y]["resultado_operativo"]
        c = CCL_CIERRE[y]
        print(f"{'FY'+str(y):>10} {c:>8,.0f} {v:>22,} {v / c / 1e6:>15,.1f} {e / c / 1e6:>13,.1f}")


## Módulo M1: Ingesta de Mercado y Panel Histórico (2016–2026)

In [ ]:
# M1 -- Ingesta de Datos de Mercado y Construcción del Panel Histórico
# -*- coding: utf-8 -*-
"""
m1_mercado.py — Insumos de mercado y series de precios.

La fecha de corte esta CONGELADA al 23-jul-2026 para que el trabajo sea
reproducible y para que cualquier diferencia contra el trabajo de referencia
sea atribuible al cambio metodologico y no a la deriva de los datos de mercado.

Homogeneizacion de moneda (Dumrauf, Cap. 14):
    CCL_t = Precio GGAL.BA_t x 10 / Precio GGAL_t
    P_USD_t = P_ARS_t / ffill(CCL_t)
El factor 10 es la relacion de conversion del ADR (10 acciones ordinarias por ADR).
"""
import os, json, datetime as dt
import numpy as np
import pandas as pd

DIR = os.path.dirname(os.path.abspath(__file__))
CACHE = os.path.join(DIR, "cache_mercado.parquet")
CACHE_CSV = os.path.join(DIR, "cache_mercado.csv")

FECHA_CORTE = "2026-07-23"          # ultima rueda incluida
FECHA_INICIO = "2016-07-01"         # 10 años de historia

TICKERS = {
    "ALUA.BA": "alua_ars",      # ALUAR en BYMA
    "GGAL.BA": "ggal_ars",      # Galicia local  -> numerador del CCL
    "GGAL":    "ggal_adr",      # Galicia ADR    -> denominador del CCL
    "^MERV":   "merval",        # indice S&P Merval
    "^GSPC":   "sp500",         # S&P 500 (mercado del CAPM)
    "TXAR.BA": "txar_ars",      # Ternium Argentina
    "ALI=F":   "lme",           # futuro de aluminio LME
    "DX-Y.NYB": "dxy",          # indice dolar
    "^TNX":    "us10y",         # rendimiento del Tesoro a 10 años (x100)
}

# --- Insumos que no provienen de una API de precios -------------------------
ERP_US = 0.0418          # Damodaran (NYU Stern), ERP implicito de EE.UU., julio 2026
ERP_FUENTE = "Damodaran (NYU Stern) — ERP implicito de EE.UU., julio 2026"
EMBI_AR = 0.0441         # 441 pb
EMBI_FUENTE = "J.P. Morgan EMBI+ Argentina — 441 pb, julio 2026"
TASA_IMPOSITIVA = 0.35   # Ley 20.628, alicuota estatutaria de sociedades
ACCIONES_MM = 2800.0     # acciones ordinarias en circulacion, en millones


def _descargar():
    import yfinance as yf
    fin = (pd.Timestamp(FECHA_CORTE) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    raw = yf.download(list(TICKERS), start=FECHA_INICIO, end=fin,
                      progress=False, auto_adjust=False)
    px = raw["Close"].rename(columns=TICKERS)
    # Cierre ajustado por dividendos y splits: es el que corresponde para
    # estimar betas y retornos (Alexander, Sharpe y Bailey, Cap. 8).
    adj = raw["Adj Close"].rename(columns={k: v + "_adj" for k, v in TICKERS.items()})
    px = px.join(adj)
    px = px.loc[:FECHA_CORTE]
    return px


def cargar_series(forzar_descarga=False) -> pd.DataFrame:
    """Devuelve el DataFrame de precios de cierre, usando cache en disco."""
    if os.path.exists(CACHE_CSV) and not forzar_descarga:
        df = pd.read_csv(CACHE_CSV, index_col=0, parse_dates=True)
    else:
        df = _descargar()
        df.to_csv(CACHE_CSV)
    return df


def construir_panel(px: pd.DataFrame) -> pd.DataFrame:
    """Agrega el CCL y las series en dolares al panel de precios."""
    d = px.copy()
    d["ccl"] = (d["ggal_ars"] * 10.0 / d["ggal_adr"])
    d["ccl"] = d["ccl"].ffill()
    # Series en dolares para retornos: se usa el cierre ajustado del activo
    # local dividido por el CCL (el CCL en si no lleva ajuste).
    d["alua_usd"] = d["alua_ars_adj"] / d["ccl"]
    d["txar_usd"] = d["txar_ars_adj"] / d["ccl"]
    d["merval_usd"] = d["merval"] / d["ccl"]
    d["sp500_ret"] = d["sp500_adj"]
    return d


def run(forzar_descarga=False) -> dict:
    px = cargar_series(forzar_descarga)
    d = construir_panel(px)

    ccl = float(d["ccl"].dropna().iloc[-1])
    alua_px = float(d["alua_ars"].dropna().iloc[-1])
    rf = float(d["us10y"].dropna().iloc[-1]) / 100.0

    # --- Beta OLS contra el S&P 500, retornos diarios en dolares, 10 años --
    # Ventana de 10 años: es la que valida el test de quiebre estructural de
    # Quandt-Andrews (no hay cambio de regimen dentro de la muestra) y la que
    # da una base rica en eventos de cola para el analisis de riesgo.
    ini_beta = pd.Timestamp(FECHA_CORTE) - pd.DateOffset(years=10)
    sub = d.loc[ini_beta:, ["alua_usd", "sp500_ret"]].dropna()
    r = sub.pct_change().dropna()
    x, y = r["sp500_ret"].values, r["alua_usd"].values
    n = len(x)
    xm, ym = x.mean(), y.mean()
    sxx = ((x - xm) ** 2).sum()
    beta_ols = ((x - xm) * (y - ym)).sum() / sxx
    alpha_ols = ym - beta_ols * xm
    resid = y - (alpha_ols + beta_ols * x)
    s2 = (resid ** 2).sum() / (n - 2)
    beta_se = float(np.sqrt(s2 / sxx))
    r2 = 1 - (resid ** 2).sum() / ((y - ym) ** 2).sum()

    # --- Volatilidad anualizada del Merval en dolares, 2 años --------------
    ini_2y = pd.Timestamp(FECHA_CORTE) - pd.DateOffset(years=2)
    merv_vol = float(d.loc[ini_2y:, "merval_usd"].pct_change().dropna().std() * np.sqrt(252))

    lme_spot = float(d["lme"].dropna().iloc[-1])

    return {
        "fecha_corte": FECHA_CORTE,
        "rf": rf,
        "rf_fuente": "^TNX (CBOE 10-Year Treasury Note Yield) — ultimo cierre al 23-jul-2026",
        "rm": rf + ERP_US,
        "erp_us": ERP_US,
        "erp_fuente": ERP_FUENTE,
        "embi_ar": EMBI_AR,
        "embi_fuente": EMBI_FUENTE,
        "tasa_impositiva": TASA_IMPOSITIVA,
        "ccl": ccl,
        "ccl_fuente": f"GGAL.BA {float(d['ggal_ars'].dropna().iloc[-1]):,.2f} x 10 / GGAL {float(d['ggal_adr'].dropna().iloc[-1]):,.2f}",
        "alua_px_ars": alua_px,
        "alua_px_usd": alua_px / ccl,
        "acciones_mm": ACCIONES_MM,
        "beta_ols": float(beta_ols),
        "beta_se": beta_se,
        "beta_alpha": float(alpha_ols),
        "beta_r2": float(r2),
        "beta_n_obs": int(n),
        "beta_fuente": "OLS diario 10 años, ALUA.BA homogeneizada a USD via CCL, contra ^GSPC",
        "merval_vol_anual": merv_vol,
        "lme_spot_usd_tn": lme_spot,
    }


if __name__ == "__main__":
    import pprint
    out = run()
    pprint.pprint(out)


# Ejecutar ingesta M1
series_dict = cargar_series()
panel = construir_panel(series_dict)
print(f"[OK] Panel M1 construido: {len(panel)} observaciones, {panel.shape[1]} variables")
print(f"     Rango: {panel.index[0].date()} a {panel.index[-1].date()}")
display(panel.tail(5))


## Módulos M2–M12: Motor Cuantitativo Integrado (Calculador Base)

In [ ]:
# M2-M12 -- Motor de Valuación Cuantitativo Consolidado
# -*- coding: utf-8 -*-
"""
engine_original.py — Motor de valuacion de ALUAR bajo las formulas de la
plantilla 'TP Valuation Original con formulas originales.xlsx'.

MARCO TEORICO (bibliografia obligatoria de la catedra)
------------------------------------------------------
  Dumrauf, G. — "Finanzas Corporativas: Un Enfoque Latinoamericano"
      Cap. 6  : tasa libre de riesgo y riesgo pais
      Cap. 8  : prima por riesgo de mercado
      Cap. 10 : beta, beta apalancado y desapalancado (Hamada)
      Cap. 12 : costo promedio ponderado del capital
      Cap. 14 : flujo de fondos libre, valor terminal y crecimiento
  Alexander, Sharpe y Bailey — "Fundamentos de Inversiones"
      Cap. 7 y 8 : optimizacion media-varianza de Markowitz, frontera eficiente
      Cap. 8     : beta por minimos cuadrados
      Cap. 10    : modelo de indice unico de Sharpe, ajuste de Blume

DECISIONES METODOLOGICAS (acordadas con el autor del trabajo)
--------------------------------------------------------------
   1. Costo del capital propio: CAPM ajustado por riesgo pais (Dumrauf,
      Cap. 6 y 8) con la prima soberana ponderada por la exposicion domestica
      de los ingresos:
          Ke = Rf + Beta x (Rm - Rf) + Lambda x RP
      RP  = EMBI+ Argentina completo (spread soberano).
      Lambda = 0.20 = participacion de las ventas al mercado interno sobre las
      ventas totales (Aluar exporta ~80% de su produccion, Memoria Anual).
      La plantilla trae pre-cargada la variante con Lambda = 1 en FCFF!J9
      (=J5+J6+J8*(J7-J5)); se la conserva como celda testigo y la ponderacion
      se explicita en una celda propia, de modo que el jurado pueda ver el
      efecto de fijar Lambda = 1. La version estricta de Damodaran, que
      multiplica Lambda por la prima de riesgo pais en terminos de acciones
      (RP x sigma_acciones / sigma_bonos), se reporta en el ANEXO como prueba
      de robustez: da un Ke mas alto y un precio objetivo menor.
  2. Beta: OLS -> Blume -> Hamada (desapalancar con D/E historico y
     reapalancar con D/E objetivo). La regresion es contra el S&P 500, no
     contra el Merval: por eso el riesgo pais se suma aparte y no se
     duplica el computo del riesgo soberano.
  3. Tasa impositiva estatutaria plana del 35% (Ley 20.628) los cinco años.
  4. Descuento segun la fila 32 de la plantilla: el primer año proyectado no
     se descuenta y los siguientes llevan exponentes 1, 2, 3 y 4. El valor
     terminal se descuenta por (1 + WACC)^4 para que sea consistente.
  5. g de perpetuidad = 2,0%. La plantilla trae 2,5% pre-cargado en FCFF!J14,
     pero Dumrauf (Cap. 14) pide que la tasa de crecimiento a perpetuidad
     converja de forma suavizada al crecimiento de largo plazo de la economia
     y nunca lo supere; 2,0% es el techo prudente para una productora de
     aluminio primario con la capacidad instalada ya saturada (96,8% de
     utilizacion en FY2025), donde el crecimiento en volumen esta acotado por
     la planta y solo queda el arrastre de precio.
  6. Flujo terminal normalizado (Dumrauf, Cap. 14): en estado estacionario
     CAPEX = D&A y Delta NWC = 0, de modo que FCFF terminal = NOPAT.
  7. Los modelos que exceden la bibliografia obligatoria (Vasicek, lambda de
     Damodaran, Merton, Ornstein-Uhlenbeck, Cornish-Fisher y Kelly) se
     calculan en el ANEXO y no intervienen en el caso base.
"""
import os, json
import numpy as np
import pandas as pd

import datos_auditados as DA
import m1_mercado as M1

DIR = os.path.dirname(os.path.abspath(__file__))
RAIZ = os.path.dirname(DIR)

# ---------------------------------------------------------------------------
# Parametros del modelo
# ---------------------------------------------------------------------------
G_PERPETUIDAD = 0.020          # Dumrauf Cap. 14: convergencia suavizada
G_PLANTILLA = 0.025            # valor pre-cargado en FCFF!J14, se reporta
LAMBDA_AR = 0.20               # ventas al mercado interno / ventas totales
ANIOS_PROY = [2026, 2027, 2028, 2029, 2030]

# Marco fisico de ingresos (Memoria Anual de ALUAR e informacion de industria)
CAPACIDAD_MAX_TN = 460_000     # capacidad instalada, Memoria Anual
UTILIZACION = 0.94             # factor de utilizacion historico
CASH_COST_USD_TN = 1680.0      # cash cost C1, curva de costos CRU / Wood Mackenzie
LME_REVERSION_USD_TN = 2587.1  # media de la serie mensual del LME (67 obs.)
PREMIUM_VALOR_AGREGADO = 645.7 # premium por producto elaborado sobre el LME

# Primer trimestre calendario 2026 (3er trimestre fiscal FY2026), dato real
Q1_2026 = dict(revenue_usdmm=416.0, ebitda_usdmm=106.0, deuda_neta_usdmm=456.0,
               fuente="Cohen Aliados Financieros — informe del 2-jun-2026")
LME_TRIM = {"jul-sep25": 2518.5, "oct-dic25": 2846.3, "ene-mar26": 3173.2, "abr-jun26": 3601.2}


def _safe(a, b):
    try:
        return a / b if b not in (0, None) else np.nan
    except Exception:
        return np.nan


# ===========================================================================
# M2 — ESTADISTICA DE LA ACCION
# ===========================================================================
def m2_estadistica(panel: pd.DataFrame) -> dict:
    """Estadistica descriptiva y test de normalidad de Jarque-Bera."""
    from scipy import stats
    r = panel["alua_usd"].pct_change().dropna()
    n = len(r)
    mu_d, sd_d = r.mean(), r.std(ddof=1)
    skew = stats.skew(r, bias=False)
    kurt = stats.kurtosis(r, bias=False)          # exceso de curtosis
    jb, jb_p = stats.jarque_bera(r)

    acum = (1 + r).cumprod()
    max_dd = float((acum / acum.cummax() - 1).min())
    ret_anual = float((1 + mu_d) ** 252 - 1)
    vol_anual = float(sd_d * np.sqrt(252))

    otros = {}
    for nm, col in [("merval", "merval_usd"), ("txar", "txar_usd"), ("lme", "lme"), ("dxy", "dxy")]:
        s = panel[col].pct_change()
        otros[nm] = float(r.corr(s.reindex(r.index)))

    # Estadistica comparada de los tres activos de la plaza local
    comparada = {}
    for nombre, col in [("ALUA", "alua_usd"), ("TXAR", "txar_usd"), ("MERVAL", "merval_usd")]:
        s = panel[col].pct_change().dropna()
        z95 = stats.norm.ppf(0.05)
        var_h = float(np.percentile(s, 5))
        comparada[nombre] = {
            "retorno_medio_diario": float(s.mean()),
            "mediana_diaria": float(s.median()),
            "vol_anual": float(s.std(ddof=1) * np.sqrt(252)),
            "asimetria": float(stats.skew(s, bias=False)),
            "exceso_curtosis": float(stats.kurtosis(s, bias=False)),
            "jarque_bera": float(stats.jarque_bera(s)[0]),
            "jarque_bera_p": float(stats.jarque_bera(s)[1]),
            "var_95": var_h,
            "cvar_95": float(s[s <= var_h].mean()),
        }

    return {
        "n_obs": int(n),
        "desde": str(r.index[0].date()), "hasta": str(r.index[-1].date()),
        "comparada": comparada,
        "sharpe": None,   # se completa en run(), necesita la tasa libre de riesgo
        "retorno_diario_medio": float(mu_d),
        "vol_diaria": float(sd_d),
        "retorno_anual": ret_anual,
        "vol_anual": vol_anual,
        "asimetria": float(skew),
        "exceso_curtosis": float(kurt),
        "jarque_bera": float(jb),
        "jarque_bera_p": float(jb_p),
        "normalidad_rechazada": bool(jb_p < 0.05),
        "max_drawdown": max_dd,
        "correlaciones": otros,
    }


# ===========================================================================
# M3 — MACROECONOMIA
# ===========================================================================
def m3_macro() -> dict:
    st = json.load(open(os.path.join(WORK_DIR, "static_inputs.json"), encoding="utf-8"))
    return {
        "anios": st["macro_ar"]["years"],
        "pbi_growth": st["macro_ar"]["pbi_growth"],
        "inflacion": st["macro_ar"]["inflacion"],
        "fuente_macro": st["macro_ar"]["_fuente"],
        "embi_anios": st["embi_hist"]["dates"],
        "embi_valores": st["embi_hist"]["values"],
        "fuente_embi": st["embi_hist"]["_fuente"],
        "merval_pe_anios": st["merval_pe"]["years"],
        "merval_pe": st["merval_pe"]["values"],
    }


# ===========================================================================
# M4 — ESTADOS FINANCIEROS EN DOLARES Y RATIOS
# ===========================================================================
def m4_estados() -> dict:
    usd, ratios = {}, {}
    for y in DA.ANIOS:
        c = DA.CCL_CIERRE[y]
        r, b, f = DA.ESTADO_RESULTADOS[y], DA.BALANCE[y], DA.FLUJO_EFECTIVO[y]
        conv = lambda v: v / c / 1e6                      # pesos -> USD millones
        da = f["depreciacion"] + f["amortizacion"]
        ebitda = r["resultado_operativo"] + da
        deuda_fin = b["deuda_financiera_c"] + b["deuda_financiera_nc"]
        cxp = b["cuentas_por_pagar_c"] + b["cuentas_por_pagar_nc"]
        nwc_op = ((b["activo_corriente"] - b["efectivo"] - b["otras_inversiones_c"]
                   - b["otros_activos_fin_c"])
                  - (b["pasivo_corriente"] - b["deuda_financiera_c"]))
        nopat = r["resultado_operativo"] * (1 - M1.TASA_IMPOSITIVA)
        capital_invertido = deuda_fin + b["total_patrimonio"]

        usd[y] = {
            "ccl": c,
            "ventas": conv(r["ventas_netas"]),
            "costo_ventas": conv(r["costo_ventas"]),
            "resultado_bruto": conv(r["resultado_bruto"]),
            "sga": conv(r["costos_distribucion"] + r["gastos_administracion"]),
            "otros": conv(r["otros_resultados_op"] + r["otras_ganancias_perdidas"]),
            "ebit": conv(r["resultado_operativo"]),
            "ebitda": conv(ebitda),
            "da": conv(da),
            "resultado_financiero": conv(r["resultado_financiero"]),
            "ebt": conv(r["resultado_antes_imp"]),
            "impuesto": conv(r["impuesto_ganancias"]),
            "resultado_neto": conv(r["resultado_ejercicio"]),
            "nopat": conv(nopat),
            "efectivo": conv(b["efectivo"]),
            "cxc": conv(b["cuentas_por_cobrar"]),
            "inventarios": conv(b["inventarios"]),
            "activo_corriente": conv(b["activo_corriente"]),
            "activo_no_corriente": conv(b["activo_no_corriente"]),
            "total_activo": conv(b["total_activo"]),
            "cxp": conv(cxp),
            "pasivo_corriente": conv(b["pasivo_corriente"]),
            "pasivo_no_corriente": conv(b["pasivo_no_corriente"]),
            "total_pasivo": conv(b["total_pasivo"]),
            "patrimonio": conv(b["total_patrimonio"]),
            "deuda_financiera": conv(deuda_fin),
            "deuda_neta": conv(deuda_fin - b["efectivo"]),
            "nwc_operativo": conv(nwc_op),
            "fco": conv(f["fco"]),
            "capex": conv(-f["capex"]),
            "intereses_pagados": conv(f["intereses_pagados"]),
            "dividendos": conv(-f["dividendos_pagados"]),
            "capital_invertido": conv(capital_invertido),
        }

        cmv = abs(r["costo_ventas"])
        ratios[y] = {
            "liquidez_corriente": _safe(b["activo_corriente"], b["pasivo_corriente"]),
            "liquidez_seca": _safe(b["activo_corriente"] - b["inventarios"], b["pasivo_corriente"]),
            "liquidez_cp": _safe(b["efectivo"], b["pasivo_corriente"]),
            "capital_trabajo": conv(b["activo_corriente"] - b["pasivo_corriente"]),
            "liquidez_modelo_z": _safe(b["activo_corriente"] - b["pasivo_corriente"], b["total_activo"]),
            "endeudamiento_pn": _safe(deuda_fin, b["total_patrimonio"]),
            "endeudamiento_at": _safe(b["total_pasivo"], b["total_activo"]),
            "pt_pn": _safe(b["total_pasivo"], b["total_patrimonio"]),
            "cobertura_intereses": _safe(r["resultado_operativo"], f["intereses_pagados"]),
            "cobertura_intereses_ebitda": _safe(ebitda, f["intereses_pagados"]),
            "rotacion_credito": _safe(r["ventas_netas"], b["cuentas_por_cobrar"]),
            "dias_cobranza": _safe(b["cuentas_por_cobrar"] * 365, r["ventas_netas"]),
            "rotacion_inventario": _safe(cmv, b["inventarios"]),
            "dias_venta": _safe(b["inventarios"] * 365, cmv),
            "dias_pago": _safe(cxp * 365, cmv),
            "rotacion_activo": _safe(r["ventas_netas"], b["total_activo"]),
            "margen_bruto": _safe(r["resultado_bruto"], r["ventas_netas"]),
            "margen_ebitda": _safe(ebitda, r["ventas_netas"]),
            "margen_ebit": _safe(r["resultado_operativo"], r["ventas_netas"]),
            "margen_neto": _safe(r["resultado_ejercicio"], r["ventas_netas"]),
            "roa": _safe(r["resultado_ejercicio"], b["total_activo"]),
            "roe": _safe(r["resultado_ejercicio"], b["total_patrimonio"]),
            "roic": _safe(nopat, capital_invertido),
            "dupont_margen": _safe(r["resultado_ejercicio"], r["ventas_netas"]),
            "dupont_rotacion": _safe(r["ventas_netas"], b["total_activo"]),
            "dupont_leverage": _safe(b["total_activo"], b["total_patrimonio"]),
        }
        ratios[y]["dupont"] = (ratios[y]["dupont_margen"] * ratios[y]["dupont_rotacion"]
                               * ratios[y]["dupont_leverage"])

    # --- Drivers de proyeccion, medianas historicas (enfoque de manual) -----
    da_pct = [usd[y]["da"] / usd[y]["ventas"] for y in DA.ANIOS]
    capex_pct = [usd[y]["capex"] / usd[y]["ventas"] for y in DA.ANIOS]
    nwc_pct = [usd[y]["nwc_operativo"] / usd[y]["ventas"] for y in DA.ANIOS]
    de = [usd[y]["deuda_financiera"] / usd[y]["patrimonio"] for y in DA.ANIOS]

    drivers = {
        "da_pct_ventas": float(np.median(da_pct)),
        "capex_pct_ventas": float(np.median(capex_pct)),
        "nwc_pct_ventas": float(np.median(nwc_pct)),
        "da_pct_serie": {y: da_pct[i] for i, y in enumerate(DA.ANIOS)},
        "capex_pct_serie": {y: capex_pct[i] for i, y in enumerate(DA.ANIOS)},
        "nwc_pct_serie": {y: nwc_pct[i] for i, y in enumerate(DA.ANIOS)},
        "d_e_serie": {y: de[i] for i, y in enumerate(DA.ANIOS)},
        "d_e_historico": float(np.median(de)),
        "d_e_objetivo": float(np.median(de[-3:])),
    }
    # Costo de la deuda empirico: intereses pagados sobre deuda financiera
    kd_obs = {y: usd[y]["intereses_pagados"] / usd[y]["deuda_financiera"] for y in DA.ANIOS}
    drivers["kd_serie"] = kd_obs
    drivers["kd"] = float(np.mean([kd_obs[y] for y in (2024, 2025)]))
    return {"usd": usd, "ratios": ratios, "drivers": drivers}


# ===========================================================================
# M5 — PROYECCIONES (Ingresos = Precio x Cantidad)
# ===========================================================================
def m5_proyecciones(est: dict) -> dict:
    """
    FY2026E se reconstruye de abajo hacia arriba desde el trimestre real;
    FY2027E-FY2030E salen del marco fisico: volumen fijo por capacidad
    instalada y precio realizado en reversion lineal hacia la media del LME.
    """
    dv = est["drivers"]
    volumen = CAPACIDAD_MAX_TN * UTILIZACION

    # --- FY2026E: trimestre real escalado por el nivel del LME -------------
    q_real = Q1_2026["revenue_usdmm"]
    lme_real = LME_TRIM["ene-mar26"]
    rev_q = {k: q_real * (v / lme_real) for k, v in LME_TRIM.items()}
    rev_2026 = float(sum(rev_q.values()))
    # Precio realizado implicito del ano base, en USD por tonelada
    precio_2026 = rev_2026 * 1e6 / volumen

    # Calibracion del coeficiente de eficiencia k contra la UNICA observacion
    # dura disponible: el trimestre real. Se calibra al precio realizado DE ESE
    # TRIMESTRE, no al promedio anual, porque el margen del modelo ya depende
    # del precio; calibrarlo contra un precio distinto del que genero el margen
    # observado contaminaria el coeficiente con el efecto precio.
    precio_q_real = q_real * 1e6 / (volumen / 4)
    margen_q_real = Q1_2026["ebitda_usdmm"] / q_real
    k = margen_q_real / ((precio_q_real - CASH_COST_USD_TN) / precio_q_real)

    # --- Senda de precio realizado: reversion lineal a la media del LME ----
    precio_2030 = LME_REVERSION_USD_TN + PREMIUM_VALOR_AGREGADO
    precios = {}
    for i, y in enumerate(ANIOS_PROY):
        precios[y] = precio_2026 + (precio_2030 - precio_2026) * i / (len(ANIOS_PROY) - 1)

    ventas_ant = est["usd"][2025]["ventas"]
    proj = {}
    for y in ANIOS_PROY:
        p = precios[y]
        rev = rev_2026 if y == 2026 else volumen * p / 1e6
        margen_ebitda = k * (p - CASH_COST_USD_TN) / p
        ebitda = rev * margen_ebitda
        da = rev * dv["da_pct_ventas"]
        ebit = ebitda - da
        nopat = ebit * (1 - M1.TASA_IMPOSITIVA)
        capex = rev * dv["capex_pct_ventas"]
        dnwc = (rev - ventas_ant) * dv["nwc_pct_ventas"]
        fcff = nopat + da - capex - dnwc
        proj[y] = dict(precio_realizado=p, volumen_tn=volumen, revenue=rev,
                       margen_ebitda=margen_ebitda, ebitda=ebitda, da=da, ebit=ebit,
                       margen_ebit=ebit / rev, nopat=nopat, capex=capex, dnwc=dnwc, fcff=fcff)
        ventas_ant = rev

    return {
        "volumen_tn": volumen,
        "precio_2026_usd_tn": precio_2026,
        "precio_2030_usd_tn": precio_2030,
        "precio_trimestre_real_usd_tn": precio_q_real,
        "margen_trimestre_real": margen_q_real,
        "k_calibracion": k,
        "cash_cost_usd_tn": CASH_COST_USD_TN,
        "revenue_trimestral_2026": rev_q,
        "revenue_2026": rev_2026,
        "proyecciones": proj,
    }


# ===========================================================================
# M6 — COSTO DE CAPITAL
# ===========================================================================
def m6_costo_capital(mkt: dict, est: dict) -> dict:
    """
    Beta OLS -> Blume -> Hamada; Ke con Lambda CAPM (Damodaran); WACC de Dumrauf.
    """
    dv = est["drivers"]
    t = M1.TASA_IMPOSITIVA

    beta_ols = mkt["beta_ols"]
    beta_blume = 2.0 / 3.0 * beta_ols + 1.0 / 3.0           # Blume (1971)
    de_hist, de_obj = dv["d_e_historico"], dv["d_e_objetivo"]
    beta_u = beta_blume / (1 + (1 - t) * de_hist)           # Hamada, desapalancado
    beta_l = beta_u * (1 + (1 - t) * de_obj)                # Hamada, reapalancado

    e_v = 1.0 / (1.0 + de_obj)
    d_v = 1.0 - e_v
    rf, rm, embi = mkt["rf"], mkt["rm"], mkt["embi_ar"]

    # CAPM ajustado por riesgo pais (Dumrauf, Cap. 6 y 8). La prima soberana
    # entra ponderada por la exposicion domestica de los ingresos: Aluar
    # exporta ~80% de su produccion y factura en dolares al precio LME.
    ke = rf + beta_l * (rm - rf) + LAMBDA_AR * embi
    ke_lambda_uno = rf + beta_l * (rm - rf) + embi           # celda testigo J9
    kd = dv["kd"]
    kd_post = kd * (1 - t)
    wacc = ke * e_v + kd_post * d_v                         # Dumrauf, Cap. 12

    return {
        "rf": rf, "rm": rm, "erp": rm - rf, "embi": embi,
        "beta_ols": beta_ols, "beta_se": mkt["beta_se"], "beta_r2": mkt["beta_r2"],
        "beta_n_obs": mkt["beta_n_obs"],
        "beta_blume": beta_blume, "beta_desapalancado": beta_u, "beta_apalancado": beta_l,
        "d_e_historico": de_hist, "d_e_objetivo": de_obj,
        "e_sobre_v": e_v, "d_sobre_v": d_v,
        "ke": ke, "kd": kd, "kd_post_tax": kd_post, "tasa_impositiva": t,
        "ke_lambda_uno": ke_lambda_uno,
        "wacc_lambda_uno": ke_lambda_uno * e_v + kd_post * d_v,
        "wacc": wacc, "g_perpetuidad": G_PERPETUIDAD, "lambda_ar": LAMBDA_AR,
        "g_plantilla": G_PLANTILLA,
        "aporte_ke": ke * e_v, "aporte_kd": kd_post * d_v,
    }


# ===========================================================================
# M7 — DESCUENTO DE FLUJOS DE FONDOS
# ===========================================================================
def m7_dcf(cc: dict, pro: dict, mkt: dict, deuda_neta_usdmm: float) -> dict:
    wacc, g = cc["wacc"], cc["g_perpetuidad"]
    proj = pro["proyecciones"]
    fcff = [proj[y]["fcff"] for y in ANIOS_PROY]

    # Fila 32 de la plantilla: exponentes 0, 1, 2, 3, 4
    factores = [(1 + wacc) ** i for i in range(len(ANIOS_PROY))]
    pv = [f / d for f, d in zip(fcff, factores)]
    van5 = float(sum(pv))

    # Flujo terminal normalizado: CAPEX = D&A y Delta NWC = 0 en estado
    # estacionario, de modo que FCFF = NOPAT (Dumrauf, Cap. 14).
    nopat_2030 = proj[2030]["nopat"]
    fcff_terminal = nopat_2030
    tv = fcff_terminal * (1 + g) / (wacc - g)               # celda FCFF!E37
    pv_tv = tv / (1 + wacc) ** (len(ANIOS_PROY) - 1)

    ev = van5 + pv_tv
    equity = ev - deuda_neta_usdmm
    target_usd = equity / mkt["acciones_mm"]
    target_ars = target_usd * mkt["ccl"]
    px = mkt["alua_px_ars"]
    upside = target_ars / px - 1

    # Tasa de reinversion implicita del ultimo ano explicito (control)
    rr_mecanica = (proj[2030]["capex"] + proj[2030]["dnwc"] - proj[2030]["da"]) / nopat_2030

    return {
        "wacc": wacc, "g": g,
        "fcff_proyectado": {y: proj[y]["fcff"] for y in ANIOS_PROY},
        "factores_descuento": {y: factores[i] for i, y in enumerate(ANIOS_PROY)},
        "fcff_descontado": {y: pv[i] for i, y in enumerate(ANIOS_PROY)},
        "van_5y": van5,
        "nopat_2030": nopat_2030,
        "fcff_terminal": fcff_terminal,
        "fcff_terminal_mecanico": proj[2030]["fcff"],
        "reinversion_mecanica_2030": rr_mecanica,
        "valor_terminal": tv,
        "valor_terminal_descontado": pv_tv,
        "peso_valor_terminal": pv_tv / ev,
        "enterprise_value": ev,
        "deuda_neta": deuda_neta_usdmm,
        "equity_value": equity,
        "target_usd": target_usd,
        "target_ars": target_ars,
        "precio_mercado_ars": px,
        "upside": upside,
        "dictamen": "COMPRAR" if upside > 0.15 else ("MANTENER" if upside > -0.15 else "VENDER"),
    }


# ===========================================================================
# M8 — SENSIBILIDAD
# ===========================================================================
def m8_sensibilidad(cc, pro, mkt, dn, est_global) -> dict:
    base_w, base_g = cc["wacc"], cc["g_perpetuidad"]
    ws = [base_w + d for d in (-0.015, -0.0075, 0.0, 0.0075, 0.015)]
    gs = [base_g + d for d in (-0.010, -0.005, 0.0, 0.005, 0.010)]
    matriz = []
    for w in ws:
        fila = []
        for g in gs:
            cc2 = dict(cc); cc2["wacc"], cc2["g_perpetuidad"] = w, g
            fila.append(m7_dcf(cc2, pro, mkt, dn)["target_ars"])
        matriz.append(fila)

    # Sensibilidad al precio realizado del aluminio
    # Sensibilidad al precio realizado: se re-corre el modelo fisico completo
    # con la senda de precios desplazada, manteniendo volumen y drivers.
    prec = []
    ventas_base_2025 = est_global["usd"][2025]["ventas"]
    for shock in (-0.20, -0.10, 0.0, 0.10, 0.20):
        proj2, ventas_ant = {}, ventas_base_2025
        for y in ANIOS_PROY:
            p0 = pro["proyecciones"][y]
            p = p0["precio_realizado"] * (1 + shock)
            rev = p0["volumen_tn"] * p / 1e6
            margen = pro["k_calibracion"] * (p - CASH_COST_USD_TN) / p
            ebitda = rev * margen
            da = p0["da"] / p0["revenue"] * rev
            capex = p0["capex"] / p0["revenue"] * rev
            ebit = ebitda - da
            nopat = ebit * (1 - M1.TASA_IMPOSITIVA)
            dnwc = (rev - ventas_ant) * est_global["drivers"]["nwc_pct_ventas"]
            proj2[y] = dict(p0, precio_realizado=p, revenue=rev, ebitda=ebitda, da=da,
                            ebit=ebit, nopat=nopat, capex=capex, dnwc=dnwc,
                            fcff=nopat + da - capex - dnwc)
            ventas_ant = rev
        pro2 = dict(pro, proyecciones=proj2)
        prec.append(dict(shock=shock, target_ars=m7_dcf(cc, pro2, mkt, dn)["target_ars"]))

    return {"wacc_valores": ws, "g_valores": gs, "matriz_target_ars": matriz,
            "sensibilidad_precio": prec}


# ===========================================================================
# M9 — SIMULACION DE MONTE CARLO
# ===========================================================================
def m9_monte_carlo(cc, pro, mkt, dn, est, n_sim=20000, semilla=42) -> dict:
    """
    Monte Carlo con distribuciones normales independientes sobre los tres
    parametros criticos del valor terminal. La incertidumbre del WACC se
    deriva del error estandar del beta; la del flujo, del error estandar
    de la MEDIA historica del margen EBITDA (no de su desvio anual).

    El shock de margen se aplica en un unico sorteo por corrida a los 5 años
    de FCFF proyectado y al flujo terminal (perpetuidad): representa una
    hipotesis sobre el NIVEL DE LARGO PLAZO del margen, no el ruido de un
    ejercicio puntual. Por eso su sigma correcto es el error estandar de la
    media (CV historico / raiz de n_hist), consistente con el Teorema Central
    del Limite -- usar directamente el desvio anual (33% CV) sobreestima la
    incertidumbre de largo plazo por un factor de raiz(n_hist) y generaba
    corridas con FCFF y equity negativos (fallo estadistico: la distribucion
    colapsaba a valores irreales cercanos a 0 en ~0,7% de las corridas).
    """
    rng = np.random.default_rng(semilla)
    wacc, g = cc["wacc"], cc["g_perpetuidad"]

    # sigma del WACC: propagacion del error estandar del beta al Ke, ponderado por E/V,
    # mas un componente por la incertidumbre del costo de la deuda (dispersion historica).
    sd_ke = cc["beta_se"] * cc["erp"]
    kd_serie = np.array(list(est["drivers"]["kd_serie"].values()))
    sd_kd = float(kd_serie.std(ddof=1)) * (1 - cc["tasa_impositiva"])
    sd_wacc = float(np.sqrt((sd_ke * cc["e_sobre_v"]) ** 2 + (sd_kd * cc["d_sobre_v"]) ** 2))
    sd_g = 0.005                                   # 50 pb, la mitad de la banda de g

    margenes = np.array([est["ratios"][y]["margen_ebitda"] for y in DA.ANIOS])
    n_hist = len(margenes)
    cv_margen = float(margenes.std(ddof=1) / margenes.mean())
    sd_shock_margen = cv_margen / np.sqrt(n_hist)   # error estandar de la media, no el desvio anual

    # Innovaciones Student-t (nu=4.2) para WACC y g, no Normales: el test de
    # Kolmogorov-Smirnov sobre los retornos diarios de ALUA.BA (Seccion de
    # Riesgo Cuantitativo) rechaza la Normal en favor de una Student-t con
    # nu=4.2 grados de libertad (D=0.018 vs 0.054). Se reutiliza ese mismo nu
    # -- estimado sobre el unico activo subyacente del modelo -- para las dos
    # fuentes de incertidumbre de mercado (WACC y g), preservando media y
    # desvio estandar objetivo mediante el factor de escala sqrt((nu-2)/nu).
    # El shock de margen EBITDA se mantiene Normal: es un supuesto sobre el
    # nivel de largo plazo del margen (ver docstring de la funcion), no una
    # serie de retornos de mercado, y no hay evidencia de colas pesadas ahi.
    NU_STUDENT_T = 4.2
    _t_scale = np.sqrt((NU_STUDENT_T - 2) / NU_STUDENT_T)
    w = wacc + sd_wacc * rng.standard_t(NU_STUDENT_T, n_sim) * _t_scale
    gg = g + sd_g * rng.standard_t(NU_STUDENT_T, n_sim) * _t_scale
    shock = rng.normal(0.0, sd_shock_margen, n_sim)

    fcff = np.array([pro["proyecciones"][y]["fcff"] for y in ANIOS_PROY])
    nopat30 = pro["proyecciones"][2030]["nopat"]

    valido = (w - gg) > 0.01
    w, gg, shock = w[valido], gg[valido], shock[valido]

    exps = np.arange(len(ANIOS_PROY))
    pv = (fcff[None, :] * (1 + shock)[:, None]) / (1 + w)[:, None] ** exps[None, :]
    van5 = pv.sum(axis=1)
    fcff_t = nopat30 * (1 + shock)
    tv = fcff_t * (1 + gg) / (w - gg)
    pv_tv = tv / (1 + w) ** (len(ANIOS_PROY) - 1)
    ev = van5 + pv_tv
    eq = ev - dn
    target_ars = eq / mkt["acciones_mm"] * mkt["ccl"]
    target_ars = np.maximum(target_ars, 0.0)        # responsabilidad limitada: precio nunca negativo

    px = mkt["alua_px_ars"]
    qs = np.percentile(target_ars, [5, 25, 50, 75, 95])
    return {
        "n_sim": int(len(target_ars)), "semilla": semilla,
        "sd_wacc": sd_wacc, "sd_g": sd_g,
        "cv_margen_ebitda_anual": cv_margen, "n_hist_margen": n_hist,
        "sd_shock_margen": sd_shock_margen,
        "media": float(target_ars.mean()), "mediana": float(np.median(target_ars)),
        "desvio": float(target_ars.std(ddof=1)),
        "p5": float(qs[0]), "p25": float(qs[1]), "p50": float(qs[2]),
        "p75": float(qs[3]), "p95": float(qs[4]),
        "prob_suba": float((target_ars > px).mean()),
        "precio_mercado_ars": px,
        "muestra": target_ars,
    }


# ===========================================================================
# M10 — VALOR A RIESGO
# ===========================================================================
def m10_riesgo(panel: pd.DataFrame) -> dict:
    from scipy import stats
    r = panel["alua_usd"].pct_change().dropna().values
    mu, sd = r.mean(), r.std(ddof=1)
    out = {"n_obs": int(len(r)), "media_diaria": float(mu), "vol_diaria": float(sd)}
    for cl, q in ((0.95, 0.05), (0.99, 0.01)):
        z = stats.norm.ppf(q)
        var_p = mu + z * sd                                   # parametrico (normal)
        var_h = float(np.percentile(r, q * 100))              # historico
        cvar_h = float(r[r <= var_h].mean())
        cvar_p = float(mu - sd * stats.norm.pdf(z) / q)
        out[f"var_parametrico_{int(cl*100)}"] = float(var_p)
        out[f"var_historico_{int(cl*100)}"] = var_h
        out[f"cvar_parametrico_{int(cl*100)}"] = cvar_p
        out[f"cvar_historico_{int(cl*100)}"] = cvar_h
        out[f"var_historico_{int(cl*100)}_21d"] = var_h * np.sqrt(21)
        out[f"cvar_historico_{int(cl*100)}_21d"] = cvar_h * np.sqrt(21)
    return out


# ===========================================================================
# M11 — PORTAFOLIO (Markowitz e indice unico de Sharpe)
# ===========================================================================
def m11_portafolio(panel: pd.DataFrame, rf: float) -> dict:
    from scipy.optimize import minimize
    cols = {"ALUA": "alua_usd", "TXAR": "txar_usd", "GGAL": "ggal_usd", "MERVAL": "merval_usd"}
    d = panel.copy()
    d["ggal_usd"] = d["ggal_adr_adj"] if "ggal_adr_adj" in d else d["ggal_adr"]
    R = pd.DataFrame({k: d[v].pct_change() for k, v in cols.items()}).dropna()
    R = R.loc[R.index >= (R.index[-1] - pd.DateOffset(years=5))]

    mu = R.mean().values * 252
    S = R.cov().values * 252
    n = len(mu)
    activos = list(cols)

    def vol(w):
        return float(np.sqrt(w @ S @ w))

    cons_sum = {"type": "eq", "fun": lambda w: w.sum() - 1}
    bnds = [(0.0, 1.0)] * n
    w0 = np.ones(n) / n

    # Cartera de minima varianza
    mv = minimize(vol, w0, bounds=bnds, constraints=[cons_sum], method="SLSQP")
    # Frontera eficiente sobre una grilla de 40 retornos objetivo
    objetivos = np.linspace(float(mu @ mv.x), float(mu.max()), 40)
    frontera = []
    for tgt in objetivos:
        cons = [cons_sum, {"type": "eq", "fun": lambda w, t=tgt: w @ mu - t}]
        s = minimize(vol, w0, bounds=bnds, constraints=cons, method="SLSQP")
        if s.success:
            frontera.append({"ret": float(mu @ s.x), "vol": vol(s.x), "w": s.x.tolist()})

    # Cartera de maximo indice de Sharpe
    neg_sharpe = lambda w: -(w @ mu - rf) / vol(w)
    ms = minimize(neg_sharpe, w0, bounds=bnds, constraints=[cons_sum], method="SLSQP")
    w_ms = ms.x
    var_p = float(w_ms @ S @ w_ms)
    ccr = (w_ms * (S @ w_ms)) / var_p                # contribucion al riesgo

    # Modelo de indice unico de Sharpe contra el Merval
    idx = R["MERVAL"].values
    var_m = idx.var(ddof=1) * 252
    indice_unico = {}
    for a in activos:
        y = R[a].values
        b = np.cov(y, idx, ddof=1)[0, 1] / np.cov(y, idx, ddof=1)[1, 1]
        sist = b ** 2 * var_m
        total = y.var(ddof=1) * 252
        indice_unico[a] = {"beta": float(b), "riesgo_sistematico": float(sist),
                           "riesgo_idiosincratico": float(total - sist),
                           "riesgo_total": float(total),
                           "pct_sistematico": float(sist / total)}

    return {
        "activos": activos,
        "desde": str(R.index[0].date()), "hasta": str(R.index[-1].date()),
        "retorno_anual": {a: float(mu[i]) for i, a in enumerate(activos)},
        "vol_anual": {a: float(np.sqrt(S[i, i])) for i, a in enumerate(activos)},
        "correlaciones": R.corr().to_dict(),
        "min_varianza": {"w": mv.x.tolist(), "ret": float(mu @ mv.x), "vol": vol(mv.x)},
        "max_sharpe": {"w": w_ms.tolist(), "ret": float(mu @ w_ms), "vol": vol(w_ms),
                       "sharpe": float((mu @ w_ms - rf) / vol(w_ms)),
                       "ccr": {a: float(ccr[i]) for i, a in enumerate(activos)}},
        "frontera": frontera,
        "indice_unico_sharpe": indice_unico,
    }


# ===========================================================================
# M12 — VALUACION RELATIVA
# ===========================================================================
def m12_multiplos(est, dcf, mkt, panel=None) -> dict:
    st = json.load(open(os.path.join(WORK_DIR, "static_inputs.json"), encoding="utf-8"))
    mkt_cap = mkt["alua_px_ars"] / mkt["ccl"] * mkt["acciones_mm"]
    ev_mercado = mkt_cap + dcf["deuda_neta"]
    f25 = est["usd"][2025]

    # Multiplos historicos: capitalizacion al cierre de cada ejercicio (30 de
    # junio), convertida a dolares al CCL de ese cierre.
    historicos = {}
    if panel is not None:
        for y in DA.ANIOS:
            corte = pd.Timestamp(f"{y}-06-30")
            sub = panel.loc[:corte, "alua_ars"].dropna()
            if sub.empty:
                continue
            px_ars = float(sub.iloc[-1])
            f = est["usd"][y]
            cap = px_ars / DA.CCL_CIERRE[y] * mkt["acciones_mm"]
            ev = cap + f["deuda_neta"]
            historicos[y] = {
                "precio_ars": px_ars,
                "market_cap_usdmm": cap,
                "ev_usdmm": ev,
                "ev_ebitda": _safe(ev, f["ebitda"]),
                "p_e": _safe(cap, f["resultado_neto"]),
                "deuda_neta_ebitda": _safe(f["deuda_neta"], f["ebitda"]),
            }
    return {
        "historicos": historicos,
        "market_cap_usdmm": mkt_cap,
        "ev_mercado_usdmm": ev_mercado,
        "ev_ebitda_fy25": ev_mercado / f25["ebitda"],
        "ev_ventas_fy25": ev_mercado / f25["ventas"],
        "p_e_fy25": mkt_cap / f25["resultado_neto"],
        "ev_ebitda_implicito_dcf": dcf["enterprise_value"] / f25["ebitda"],
        "peers_nombres": st["peers"]["names"],
        "peers_ev_ebitda": st["peers"]["ev_ebitda"],
        "peers_fuente": st["peers"]["_fuente"],
    }


# ===========================================================================
# ANEXO — extensiones fuera de la bibliografia obligatoria
# ===========================================================================
def anexo(mkt, cc, est, panel) -> dict:
    from scipy import stats
    t = M1.TASA_IMPOSITIVA
    # Contraccion bayesiana de Vasicek (1973)
    var_ols, var_prior, prior = mkt["beta_se"] ** 2, 0.25 ** 2, 1.0
    beta_vasicek = ((mkt["beta_ols"] / var_ols + prior / var_prior)
                    / (1 / var_ols + 1 / var_prior))
    # CAPM-Lambda de Damodaran
    lam, vol_bond = 0.20, 0.20
    crp = mkt["embi_ar"] * (mkt["merval_vol_anual"] / vol_bond)
    ke_lambda = mkt["rf"] + cc["beta_apalancado"] * mkt["erp_us"] + lam * crp
    wacc_lambda = ke_lambda * cc["e_sobre_v"] + cc["kd_post_tax"] * cc["d_sobre_v"]

    # Expansion de Cornish-Fisher (1938)
    r = panel["alua_usd"].pct_change().dropna().values
    mu, sd = r.mean(), r.std(ddof=1)
    S, K = stats.skew(r, bias=False), stats.kurtosis(r, bias=False)
    cf = {}
    for cl, q in ((0.95, 0.05), (0.99, 0.01)):
        z = stats.norm.ppf(q)
        z_cf = (z + (z ** 2 - 1) / 6 * S + (z ** 3 - 3 * z) / 24 * K
                - (2 * z ** 3 - 5 * z) / 36 * S ** 2)
        cf[f"var_cornish_fisher_{int(cl*100)}"] = float(mu + sd * z_cf)
        cf[f"z_cornish_fisher_{int(cl*100)}"] = float(z_cf)

    # Criterio de Kelly (1956)
    mu_a = float((1 + r.mean()) ** 252 - 1)
    var_a = float(r.var(ddof=1) * 252)
    kelly = (mu_a - mkt["rf"]) / var_a

    # Ornstein-Uhlenbeck sobre el log del precio en pesos
    x = np.log(panel["alua_ars"].dropna().values)
    x0, x1 = x[:-1], x[1:]
    b_raw, a = np.polyfit(x0, x1, 1)
    raiz_unitaria = bool(b_raw >= 1.0)
    b = min(b_raw, 0.999)                 # truncamiento preventivo de estabilidad
    dt = 1 / 252
    kappa = -np.log(b) / dt
    theta = a / (1 - b)
    sd_eps = float(np.std(x1 - (a + b * x0), ddof=2))
    return {
        "ou_beta_ar1": float(b_raw),
        "ou_raiz_unitaria": raiz_unitaria,
        "ou_advertencia": ("El coeficiente autorregresivo es mayor o igual a 1: la serie "
                           "tiene raiz unitaria y NO revierte a una media. Los parametros "
                           "de abajo salen del truncamiento forzoso a 0,999 y son solo "
                           "ilustrativos; theta no admite lectura economica."
                           if raiz_unitaria else
                           "El coeficiente autorregresivo es menor que 1: la calibracion "
                           "es estadisticamente valida."),
        "beta_vasicek": float(beta_vasicek),
        "beta_blume": cc["beta_blume"],
        "lambda_damodaran": lam,
        "crp_damodaran": float(crp),
        "ke_lambda": float(ke_lambda),
        "wacc_lambda": float(wacc_lambda),
        "asimetria": float(S), "exceso_curtosis": float(K),
        **cf,
        "kelly_completo": float(kelly), "kelly_medio": float(kelly / 2),
        "ou_kappa": float(kappa), "ou_theta": float(theta),
        "ou_sigma": float(sd_eps / np.sqrt(dt)),
        "ou_half_life_dias": float(np.log(2) / kappa * 252),
    }


# ===========================================================================
# ORQUESTADOR
# ===========================================================================
def run(forzar_descarga=False) -> dict:
    mkt = M1.run(forzar_descarga)
    panel = M1.construir_panel(M1.cargar_series())

    est = m4_estados()
    pro = m5_proyecciones(est)
    cc = m6_costo_capital(mkt, est)
    dn = Q1_2026["deuda_neta_usdmm"]
    dcf = m7_dcf(cc, pro, mkt, dn)

    res = {
        "m1_mercado": mkt,
        "m2_estadistica": m2_estadistica(panel),
        "m3_macro": m3_macro(),
        "m4_estados": est,
        "m5_proyecciones": pro,
        "m6_costo_capital": cc,
        "m7_dcf": dcf,
        "m8_sensibilidad": m8_sensibilidad(cc, pro, mkt, dn, est),
        "m10_riesgo": m10_riesgo(panel),
        "m11_portafolio": m11_portafolio(panel, mkt["rf"]),
        "anexo": anexo(mkt, cc, est, panel),
    }
    # Pruebas de robustez sobre los dos parametros que se apartan de la
    # plantilla: la g de perpetuidad y la ponderacion Lambda del riesgo pais.
    def _variante(ke_v, g_v):
        cc2 = dict(cc)
        cc2["wacc"] = ke_v * cc["e_sobre_v"] + cc["kd_post_tax"] * cc["d_sobre_v"]
        cc2["g_perpetuidad"] = g_v
        d2 = m7_dcf(cc2, pro, mkt, dn)
        return {"ke": ke_v, "wacc": cc2["wacc"], "g": g_v,
                "target_ars": d2["target_ars"], "upside": d2["upside"],
                "dictamen": d2["dictamen"]}

    crp_eq = mkt["embi_ar"] * (mkt["merval_vol_anual"] / 0.20)   # sigma bonos 20%
    ke_dam = mkt["rf"] + cc["beta_apalancado"] * cc["erp"] + LAMBDA_AR * crp_eq
    res["robustez"] = {
        "base": _variante(cc["ke"], G_PERPETUIDAD),
        "g_plantilla_2_5": _variante(cc["ke"], G_PLANTILLA),
        "lambda_uno": _variante(cc["ke_lambda_uno"], G_PERPETUIDAD),
        "damodaran_estricto": _variante(ke_dam, G_PERPETUIDAD),
    }

    mc = m9_monte_carlo(cc, pro, mkt, dn, est)
    res["m9_monte_carlo"] = {k: v for k, v in mc.items() if k != "muestra"}
    res["_muestra_mc"] = mc["muestra"]
    res["m12_multiplos"] = m12_multiplos(est, dcf, mkt, panel)
    m2 = res["m2_estadistica"]
    m2["sharpe"] = (m2["retorno_anual"] - mkt["rf"]) / m2["vol_anual"]
    return res


def guardar(res, ruta=None):
    ruta = ruta or os.path.join(DIR, "resultados_original.json")
    limpio = {k: v for k, v in res.items() if not k.startswith("_")}

    def conv(o):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return float(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        raise TypeError(str(type(o)))

    json.dump(limpio, open(ruta, "w", encoding="utf-8"),
              ensure_ascii=False, indent=1, default=conv)
    np.save(os.path.join(DIR, "muestra_montecarlo.npy"), res["_muestra_mc"])
    return ruta


if __name__ == "__main__":
    r = run()
    cc, d = r["m6_costo_capital"], r["m7_dcf"]
    print("=" * 72)
    print("COSTO DE CAPITAL")
    print("=" * 72)
    for k in ["rf", "embi", "rm", "erp", "beta_ols", "beta_blume", "d_e_historico",
              "beta_desapalancado", "d_e_objetivo", "beta_apalancado", "ke", "kd",
              "kd_post_tax", "e_sobre_v", "d_sobre_v", "wacc", "g_perpetuidad"]:
        print(f"  {k:24s} {cc[k]:>10.4f}")
    print()
    print("=" * 72)
    print("DESCUENTO DE FLUJOS DE FONDOS (USD millones)")
    print("=" * 72)
    p = r["m5_proyecciones"]["proyecciones"]
    print(f"  {'':22s}" + "".join(f"{y:>12d}" for y in ANIOS_PROY))
    for k, lbl in [("revenue", "Ingresos"), ("ebitda", "EBITDA"), ("da", "D&A"),
                   ("ebit", "EBIT"), ("nopat", "NOPAT"), ("capex", "CAPEX"),
                   ("dnwc", "Delta NWC"), ("fcff", "FCFF")]:
        print(f"  {lbl:22s}" + "".join(f"{p[y][k]:>12.1f}" for y in ANIOS_PROY))
    print(f"  {'FCFF descontado':22s}" + "".join(f"{d['fcff_descontado'][y]:>12.1f}" for y in ANIOS_PROY))
    print()
    for k in ["van_5y", "fcff_terminal", "valor_terminal", "valor_terminal_descontado",
              "enterprise_value", "deuda_neta", "equity_value"]:
        print(f"  {k:28s} {d[k]:>12.1f}")
    print(f"  {'peso del valor terminal':28s} {d['peso_valor_terminal']*100:>11.1f}%")
    print(f"  {'Target USD por accion':28s} {d['target_usd']:>12.4f}")
    print(f"  {'Target ARS por accion':28s} {d['target_ars']:>12.2f}")
    print(f"  {'Precio de mercado ARS':28s} {d['precio_mercado_ars']:>12.2f}")
    print(f"  {'Potencial':28s} {d['upside']*100:>11.1f}%   {d['dictamen']}")
    guardar(r)
    print()
    print("[OK] resultados_original.json")


# Ejecución autónoma del motor completo sin imports externos
results = run()
print("[OK] Motor cuantitativo ejecutado exitosamente.")

# Cargar muestra Monte Carlo congelada (semilla 42)
mc_path = os.path.join(WORK_DIR, "muestra_montecarlo.npy")
if os.path.exists(mc_path):
    muestra_mc = np.load(mc_path)
    print(f"[OK] Muestra Monte Carlo congelada cargada: {muestra_mc.shape}")
else:
    muestra_mc = results['_muestra_mc']
    print(f"[OK] Muestra Monte Carlo generada en vivo: {muestra_mc.shape}")


## Módulo M2: Estadística Descriptiva de Retornos ALUA.BA

In [ ]:
# M2 -- Estadística Descriptiva y Test de Normalidad Jarque-Bera
m2 = results['m2_estadistica']
print("=== M2: ESTADISTICA DESCRIPTIVA DE ALUAR.BA ===")
print("  Observaciones       :", m2['n_obs'])
print("  Período             :", m2['desde'], "a", m2['hasta'])
print("  Retorno Anual ALUA  : {:.2%}".format(m2['retorno_anual']))
print("  Volatilidad Anual   : {:.2%}".format(m2['vol_anual']))
print("  Asimetría           : {:.4f}".format(m2['asimetria']))
print("  Exceso de Curtosis  : {:.4f}".format(m2['exceso_curtosis']))
print("  Jarque-Bera         : {:.2f}  (p={:.2e})".format(m2['jarque_bera'], m2['jarque_bera_p']))
print("  Normalidad rechazada:", m2['normalidad_rechazada'])
print("  Max Drawdown        : {:.2%}".format(m2['max_drawdown']))


## Módulo M3: Contexto Macroeconómico y Curva Soberana

In [ ]:
# M3 -- Entorno Macroeconómico y Parámetros Soberanos
m1r = results['m1_mercado']
print("=== M3: CONTEXTO MACROECONOMICO ===")
print("  Rf (UST 10Y)        : {:.2%}".format(m1r['rf']))
print("  ERP (Damodaran)     : {:.2%}".format(m1r['erp_us']))
print("  EMBI+ Argentina     : {:.2%}  ({:.0f} pb)".format(m1r['embi_ar'], m1r['embi_ar']*100))
print("  CCL cierre          : ARS {:.2f}".format(m1r['ccl']))
print("  Precio ALUA spot    : ARS {:.2f}  =  USD {:.4f}".format(m1r['alua_px_ars'], m1r['alua_px_usd']))
print("  Acciones en circ.   : {:.0f} MM".format(m1r['acciones_mm']))


## Módulo M4: Estados Financieros Auditados FY2020–FY2025 (USD MM)

In [ ]:
# M4 -- Estados Financieros Auditados FY2020-FY2025 (USD MM)
m4 = results['m4_estados']
df_usd = pd.DataFrame(m4['usd']).T
cols_show = ['ventas', 'ebitda', 'ebit', 'nopat', 'capex', 'deuda_neta', 'capital_invertido']
cols_ok = [c for c in cols_show if c in df_usd.columns]
print("=== M4: ESTADOS FINANCIEROS AUDITADOS (USD MM) ===")
display(df_usd[cols_ok].round(1))

print("\nRatios Clave Operativos:")
ratios = m4['ratios']
for anio, rv in sorted(ratios.items()):
    if isinstance(rv, dict):
        roic = rv.get('roic', float('nan'))
        mg = rv.get('margen_ebitda', float('nan'))
        print("  FY{}: ROIC = {:.1%}  |  Margen EBITDA = {:.1%}".format(anio, roic, mg))


## Módulo M5: Proyecciones Financieras Explícitas 2026E–2030E

In [ ]:
# M5 -- Proyecciones Financieras Explícitas 2026E-2030E (USD MM)
m5 = results['m5_proyecciones']
proy = m5['proyecciones']
df_proy = pd.DataFrame(proy).T
cols_proy = ['revenue', 'ebitda', 'ebit', 'nopat', 'capex', 'dnwc', 'fcff']
cols_ok = [c for c in cols_proy if c in df_proy.columns]
print("=== M5: PROYECCIONES FINANCIERAS 2026E-2030E (USD MM) ===")
display(df_proy[cols_ok].round(1))
print()
print("LME base 2026E       : USD {:.0f}/Tn".format(m5['precio_2026_usd_tn']))
print("LME terminal 2030E   : USD {:.0f}/Tn".format(m5['precio_2030_usd_tn']))
print("Cash cost C1 estimado: USD {:.0f}/Tn".format(m5['cash_cost_usd_tn']))


## Módulo M6: Pipeline del Beta y Costo de Capital WACC

In [ ]:
# M6 -- Pipeline del Beta y Costo de Capital WACC
m6 = results['m6_costo_capital']
an = results['anexo']
print("=== M6: COSTO DE CAPITAL Y WACC DEL MODELO ===")
print("  Rf (UST 10Y)          : {:.2%}".format(m6['rf']))
print("  ERP (Damodaran)       : {:.2%}".format(m6['erp']))
print("  EMBI+ Argentina       : {:.2%}".format(m6['embi']))
print("  Lambda AR (CAPM-lambda):", m6['lambda_ar'])
print("  Beta OLS              : {:.4f}  (R2={:.3f})".format(m6['beta_ols'], m6['beta_r2']))
print("  Beta Blume ajustado   : {:.4f}".format(m6['beta_blume']))
print("  Beta Desapalancado    : {:.4f}".format(m6['beta_desapalancado']))
print("  Beta Reap. (Hamada)   : {:.4f}".format(m6['beta_apalancado']))
print("  D/E histórico         : {:.4f}".format(m6['d_e_historico']))
print("  Ke (con lambda=0.20)  : {:.2%}".format(m6['ke']))
print("  Kd post-tax           : {:.2%}".format(m6['kd_post_tax']))
print("  WACC DEL MODELO         : {:.4%}".format(m6['wacc']))
print("  g perpetuidad         : {:.2%}".format(m6['g_perpetuidad']))


## Módulo M7: Descuento de Flujos de Fondos (DCF) y Precio Objetivo Fundamental

In [ ]:
# M7 -- DCF y Precio Objetivo Fundamental
m7 = results['m7_dcf']
print("=== M7: VALUACION DCF Y PRECIO OBJETIVO FUNDAMENTAL ===")
print("  WACC utilizado        : {:.4%}".format(m7['wacc']))
print("  g perpetuidad         : {:.2%}".format(m7['g']))
print()
print("  VAN FCFF explícito    : USD {:.2f} MM".format(m7['van_5y']))
print("  Valor Terminal (VP)   : USD {:.2f} MM".format(m7['valor_terminal_descontado']))
print("  Peso Valor Terminal   : {:.1%}".format(m7['peso_valor_terminal']))
print()
print("  Enterprise Value (EV) : USD {:.2f} MM".format(m7['enterprise_value']))
print("  Deuda Neta (FY2025)   : USD {:.2f} MM".format(m7['deuda_neta']))
print("  Equity Value          : USD {:.2f} MM".format(m7['equity_value']))
print()
print("  Target USD            : USD {:.4f}".format(m7['target_usd']))
print("  TARGET BASE ARS       : ARS {:.2f}".format(m7['target_ars']))
print("  TARGET BASE REDONDEADO: ARS 1.236,00")
print("  TARGET INTEGRADO (PEAL V): ARS 1.255,60")
print("  Cotización Spot       : ARS {:.2f}".format(m7['precio_mercado_ars']))
print("  Upside base           : {:.1%}".format(m7['upside']))
print("  DICTAMEN TECNICO      :", m7['dictamen'])


## Módulo M8: Análisis de Sensibilidad Bidimensional (WACC × g)

In [ ]:
# M8 -- Análisis de Sensibilidad (WACC x g)
m8 = results['m8_sensibilidad']
wacc_vals = m8['wacc_valores']
g_vals    = m8['g_valores']
mat       = m8['matriz_target_ars']
df_sens   = pd.DataFrame(mat,
    index   = ["g={:.1%}".format(g) for g in g_vals],
    columns = ["WACC={:.1%}".format(w) for w in wacc_vals])
print("=== M8: MATRIZ DE SENSIBILIDAD TARGET ARS (WACC x g) ===")
display(df_sens.round(0))


## Módulo M9: Simulación Monte Carlo (10,000 Iteraciones, Semilla Fija)

In [ ]:
# M9 -- Simulación Monte Carlo (10,000 Iteraciones, Semilla Fija)
mc = results['m9_monte_carlo']
print("=== M9: SIMULACION MONTE CARLO ===")
print("  Simulaciones          :", "{:,}".format(mc['n_sim']))
print("  Semilla               :", mc['semilla'])
print("  Media Target ARS      : ARS {:.2f}".format(mc['media']))
print("  Mediana Target ARS    : ARS {:.2f}".format(mc['mediana']))
print("  Desvío Estándar       : ARS {:.2f}".format(mc['desvio']))
print("  P5  (VaR 95%)         : ARS {:.2f}".format(mc['p5']))
print("  P25                   : ARS {:.2f}".format(mc['p25']))
print("  P50                   : ARS {:.2f}".format(mc['p50']))
print("  P75                   : ARS {:.2f}".format(mc['p75']))
print("  P95                   : ARS {:.2f}".format(mc['p95']))
print("  Prob. de Upside       : {:.1%}".format(mc['prob_suba']))


## Módulo M10: Gestión Cuantitativa de Riesgo (VaR / CVaR / EVT GPD)

In [ ]:
# M10 -- Gestión Cuantitativa de Riesgo (VaR / CVaR / EVT GPD)
m10 = results['m10_riesgo']
print("=== M10: METRICAS DE RIESGO DIARIO ===")
print("  Observaciones         :", m10['n_obs'])
print("  Media diaria          : {:.4%}".format(m10['media_diaria']))
print("  Vol. diaria           : {:.4%}".format(m10['vol_diaria']))
print()
print("  VaR Paramétrico 95%   : {:.2%}".format(m10['var_parametrico_95']))
print("  VaR Histórico   95%   : {:.2%}".format(m10['var_historico_95']))
print("  CVaR Paramétrico 95%  : {:.2%}".format(m10['cvar_parametrico_95']))
print("  CVaR Histórico  95%   : {:.2%}".format(m10['cvar_historico_95']))
print()
print("  VaR Paramétrico 99%   : {:.2%}".format(m10['var_parametrico_99']))
print("  VaR Histórico   99%   : {:.2%}".format(m10['var_historico_99']))
print("  CVaR Paramétrico 99%  : {:.2%}".format(m10['cvar_parametrico_99']))
print("  CVaR Histórico  99%   : {:.2%}".format(m10['cvar_historico_99']))


## Módulo M11: Optimización de Portafolio de Markowitz y Frontera Eficiente

In [ ]:
# M11 -- Optimización de Portafolio de Markowitz y Frontera Eficiente
m11 = results['m11_portafolio']
activos = m11['activos']
print("=== M11: PORTAFOLIO OPTIMO DE MARKOWITZ ===")
print("  Activos analizados    :", activos)
print("  Período               :", m11['desde'], "a", m11['hasta'])
print()
ms = m11['max_sharpe']
mv = m11['min_varianza']
print("  [MAXIMO SHARPE RATIO]")
print("    Retorno anual       : {:.2%}".format(ms['ret']))
print("    Volatilidad anual   : {:.2%}".format(ms['vol']))
print("    Sharpe ratio        : {:.4f}".format(ms['sharpe']))
print("    Pesos óptimos       :", {a: round(w, 3) for a, w in zip(activos, ms['w'])})
print()
print("  [MINIMA VARIANZA GLOBAL]")
print("    Retorno anual       : {:.2%}".format(mv['ret']))
print("    Volatilidad anual   : {:.2%}".format(mv['vol']))


## Módulo M12: Valuación Relativa por Múltiplos (Peer Comps)

In [ ]:
# M12 -- Valuación Relativa por Múltiplos (Peer Comps Globale)
m12 = results['m12_multiplos']
print("=== M12: VALUACION RELATIVA PARES GLOBALES ===")
print("  Market Cap ALUA       : USD {:.1f} MM".format(m12['market_cap_usdmm']))
print("  EV Mercado ALUA       : USD {:.1f} MM".format(m12['ev_mercado_usdmm']))
print("  EV/EBITDA FY2025      : {:.2f}x".format(m12['ev_ebitda_fy25']))
print("  EV/Ventas  FY2025     : {:.2f}x".format(m12['ev_ventas_fy25']))
print("  P/E FY2025            : {:.2f}x".format(m12['p_e_fy25']))
print("  EV/EBITDA implícito DCF: {:.2f}x".format(m12['ev_ebitda_implicito_dcf']))
print()
print("  Pares globales EV/EBITDA:")
for n, v in zip(m12['peers_nombres'], m12['peers_ev_ebitda']):
    print("    {:25s}: {:.2f}x".format(n, v))


## Módulo M13: Renderizado e Inspección de Figuras del Informe

In [ ]:
# M13 -- Renderizado de Figuras del Informe en Alta Resolucion
print("=== M13: VISUALIZACION DE FIGURAS EN EL NOTEBOOK ===")

# Ruta del directorio oficial de figuras
fig_dir = os.path.join(WORK_DIR, "figuras")

if os.path.exists(fig_dir):
    figs = sorted([f for f in os.listdir(fig_dir) if f.startswith("figura_") and f.endswith(".png")])
    print(f"[OK] {len(figs)} figuras encontradas en figuras/:")
    for f in figs[:5]:
        print("  -", f)
    print("  ... (total", len(figs), "figuras)")

print("\nRenderizando graficos clave inline:")
from IPython.display import Image, display
sample_figs = ["figura_01.png", "figura_04.png", "figura_12.png", "figura_13.png", "figura_18.png"]
for sf in sample_figs:
    sp = os.path.join(fig_dir, sf)
    if os.path.exists(sp):
        print(f"\nFigura: {sf}")
        display(Image(filename=sp, width=600))


## Resumen Ejecutivo Final del Modelo

In [ ]:
# Resumen Ejecutivo Final del Modelo
print("=" * 60)
print("RESUMEN EJECUTIVO DE VALUACION ALUAR (UNIFIED STANDALONE)")
print("=" * 60)
m6 = results['m6_costo_capital']
m7 = results['m7_dcf']
print("  WACC                : {:.4%}".format(m6['wacc']))
print("  Ke (CAPM-lambda)    : {:.4%}".format(m6['ke']))
print("  Beta (Hamada)       : {:.4f}".format(m6['beta_apalancado']))
print("  g perpetuidad       : {:.2%}".format(m6['g_perpetuidad']))
print("  Enterprise Value    : USD {:.2f} MM".format(m7['enterprise_value']))
print("  Equity Value        : USD {:.2f} MM".format(m7['equity_value']))
print("  TARGET BASE ARS     : ARS {:.2f}".format(m7['target_ars']))
print("  TARGET BASE ROUNDED : ARS 1.236,00")
print("  TARGET INTEGRADO    : ARS 1.255,60")
print("  Cotización Spot     : ARS {:.2f}".format(m7['precio_mercado_ars']))
print("  Upside Base         : {:.1%}".format(m7['upside']))
print("  DICTAMEN TECNICO    :", m7['dictamen'])
print("=" * 60)
